In [10]:
from pathlib import Path

print("Current working dir:")
print(Path.cwd())

Current working dir:
c:\Users\koushik\Desktop\variable-naming-service\app\services


In [11]:
from pathlib import Path

cwd = Path.cwd()

if (cwd / "data").exists():
    project_root = cwd
elif (cwd.parent / "data").exists():
    project_root = cwd.parent
elif (cwd.parent.parent / "data").exists():
    project_root = cwd.parent.parent
else:
    raise FileNotFoundError("Could not locate project root containing 'data' folder.")

print("Project root detected as:", project_root)

Project root detected as: c:\Users\koushik\Desktop\variable-naming-service


In [12]:
import json

file1_path = project_root / "data" / "standards" / "autosar" / "abbreviation.json"
file2_path = project_root / "data" / "standards" / "autosar" / "new.json"

print("Old dictionary file:", file1_path)
print("New dictionary file:", file2_path)

Old dictionary file: c:\Users\koushik\Desktop\variable-naming-service\data\standards\autosar\abbreviation.json
New dictionary file: c:\Users\koushik\Desktop\variable-naming-service\data\standards\autosar\new.json


In [13]:
with open(file1_path, "r", encoding="utf-8") as f:
    old_dict = json.load(f)

with open(file2_path, "r", encoding="utf-8") as f:
    new_dict = json.load(f)

# Normalize keys and values to lowercase
old_dict = {k.lower(): v.lower() for k, v in old_dict.items()}
new_dict = {k.lower(): v.lower() for k, v in new_dict.items()}

print("Old dictionary size:", len(old_dict))
print("New dictionary size:", len(new_dict))

Old dictionary size: 1411
New dictionary size: 41


In [14]:
from collections import defaultdict

def find_internal_conflicts(dictionary):
    value_to_words = defaultdict(list)

    for word, abbr in dictionary.items():
        value_to_words[abbr].append(word)

    conflicts = {
        abbr: words
        for abbr, words in value_to_words.items()
        if len(words) > 1
    }

    return conflicts

In [15]:
old_internal_conflicts = find_internal_conflicts(old_dict)
new_internal_conflicts = find_internal_conflicts(new_dict)

print("Internal conflicts in OLD dictionary:", len(old_internal_conflicts))
print("Internal conflicts in NEW dictionary:", len(new_internal_conflicts))

Internal conflicts in OLD dictionary: 36
Internal conflicts in NEW dictionary: 0


In [16]:
reverse_old = {abbr: word for word, abbr in old_dict.items()}

In [17]:
safe_additions = []
unchanged = []
key_conflicts = []
value_conflicts = []
deletion_requests = []

for word, abbr in new_dict.items():

    if abbr == "__MISSING__":
        if word in old_dict:
            deletion_requests.append(word)
        continue

    key_exists = word in old_dict
    value_exists = abbr in reverse_old

    if key_exists:

        if old_dict[word] == abbr:
            unchanged.append(word)

        else:
            key_conflicts.append({
                "word": word,
                "old_value": old_dict[word],
                "new_value": abbr
            })

    elif value_exists:

        value_conflicts.append({
            "word": word,
            "new_value": abbr,
            "already_used_by": reverse_old[abbr]
        })

    else:

        safe_additions.append((word, abbr))

In [21]:
print("\n================ DICTIONARY ANALYSIS REPORT ================\n")

print("Old dictionary entries:", len(old_dict))
print("New dictionary entries:", len(new_dict))


print("\n--- Same Abbreviation used for multiple words in EXISTING dictionary ---")
if old_internal_conflicts:
    for abbr, words in old_internal_conflicts.items():
        print(f"{abbr} used by {words}")
else:
    print("None")

 
print("\n--- Same Abbreviation used for multiple words in NEW dictionary ---")
if new_internal_conflicts:
    for abbr, words in new_internal_conflicts.items():
        print(f"{abbr} used by {words}")
else:
    print("None")


print("\n--- New words with no conflicts ---")
for word, abbr in safe_additions:
    print(f"{word} -> {abbr}")


print("\n--- Unchanged entries ---")
for word in unchanged:
    print(word)


print("\n--- EXISTING entries with NEW abbreviations ---")
for c in key_conflicts:
    print(f"{c['word']} : old={c['old_value']} new={c['new_value']}")


print("\n--- Abbreviation already used for different words in EXISTING dictionary ---")
for c in value_conflicts:
    print(f"{c['word']} -> {c['new_value']} already used by {c['already_used_by']}")


print("\n--- Words with no Abbreviation ---")
for word in deletion_requests:
    print(word)


print("\n============================================================")


================ DICTIONARY ANALYSIS REPORT ================

Old dictionary entries: 1411
New dictionary entries: 41

--- Same Abbreviation used for multiple words in EXISTING dictionary ---
2 used by ['2', 'to']
sum used by ['addition', 'sum']
circ used by ['circle', 'circuit']
cot used by ['coated', 'cotangent']
coll used by ['collection', 'collector']
cntr used by ['container', 'counter']
coord used by ['coordinate', 'coordinated']
deg used by ['degree', 'degrees']
dt used by ['delta time', 'drivetrain']
dep used by ['dependency', 'depression']
exp used by ['expansion', 'exponential']
ext used by ['extension', 'external']
fac used by ['factor', 'factory']
fw used by ['file writer', 'freewheeling']
gen used by ['generation', 'generic']
inc used by ['inclusion', 'increase']
lat used by ['lateral', 'latitude']
le used by ['left', 'less or equal']
mod used by ['mode', 'modulo']
opt used by ['option', 'optional']
pred used by ['predication', 'predicted', 'prediction']
prof used by ['pr